In [20]:
import pandas as pd
from IPython.display import display 
import re

In [21]:
%pip install xlrd

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
#file_path = '/Users/albertomarques/Proyectos/Bootcamp/Curso2026/Semana_2/Proyecto_Semana_2/Proyecto-Shark-Attacks/DateSet_Ataque_Tiburones.xls'
#shark_df = pd.read_excel('/Users/albertomarques/Proyectos/Bootcamp/Curso2026/Semana_2/Proyecto_Semana_2/Proyecto-Shark-Attacks/DateSet_Ataque_Tiburones.xls')

In [23]:
file_path = r'c:\Users\llam_\bootcamp\projecte\Proyecto-Shark-Attacks\DateSet_Ataque_Tiburones.xls'
shark_df = pd.read_excel(file_path)

In [24]:
shark_df.shape

(7087, 23)

In [62]:
shark_df.head()

,date,year,country,state,location,species,month
3,25th march,2026,australia,South Australia,Cape Jaffa Limestone Coast,great white shark ft,march
5,18th march,2026,usa,California,Big River Beach Mendocino County,great white shark,march
6,14th march,2026,australia,Western Australia,Exmouth,unknown shark ft,march
7,10th march,2026,australia,Western Australia,Exmouth,great white shark ft,march
11,29th january,2026,brazil,Recife,Del Chifre Beach in Olinda,unknown bull and tiger sharks frequent the area,january


In [25]:
''' 
We normalize the names of the columns in the dataset and keep the colums with valuable infomration to our proyect.
'''
shark_df.columns = shark_df.columns.str.lower().str.strip().str.replace(' ','_')
cols_to_keep = ['date', 'year', 'country', 'state', 'location', 'species']
shark_df = shark_df[cols_to_keep].copy()
shark_df.head()

,date,year,country,state,location,species
0,14/04,2026.0,Maldives,Gaafu Alif Atoll,Kooddoo,Unknown
1,3rd April,2026.0,Australia,South Australia,Middleton Beach Fleurieu Peninsula Adelaide,Bronze Whaler
2,26th March,2026.0,Bahamas,Andros Island,Fresh Creek,Unknown
3,25th March,2026.0,Australia,South Australia,Cape Jaffa Limestone Coast,Great White Shark 3.5m (11.5ft)
4,22nd-23rd March,2026.0,Australia,NSW,Little Avalon Beach,Unknown


In [26]:
'''
Since the dataset covers an extensive period, we have decided to keep only the records from the last 26 years.
(roughly the average lifespan of a shark)
'''

def filter_years(df):
    """
    Filters the dataframe to keep only rows where the year 
    is between 2000 and 2026 (inclusive).
    """
    # We define the condition: year must be >= 2021 AND <= 2026
    df = df[(df['year'] >= 2000) & (df['year'] <= 2026)]
    
    # It is a good practice to reset the index after filtering rows
    return df.reset_index(drop=True)
# Apply the function
shark_df = filter_years(shark_df)


In [27]:
shark_df['year'] = shark_df['year'].astype('Int64') 
shark_df.head()

,date,year,country,state,location,species
0,14/04,2026,Maldives,Gaafu Alif Atoll,Kooddoo,Unknown
1,3rd April,2026,Australia,South Australia,Middleton Beach Fleurieu Peninsula Adelaide,Bronze Whaler
2,26th March,2026,Bahamas,Andros Island,Fresh Creek,Unknown
3,25th March,2026,Australia,South Australia,Cape Jaffa Limestone Coast,Great White Shark 3.5m (11.5ft)
4,22nd-23rd March,2026,Australia,NSW,Little Avalon Beach,Unknown


In [28]:
'''
knowing the exact location where the atack was reported it's importan to our implementation.
Wee only want to keep data corresponding to column country if we can to track them to an specific state and location.
'''
shark_df['country'] = shark_df['country'].str.lower().str.strip()
shark_df = shark_df[shark_df['state'].notnull() & shark_df['location'].notnull()]

In [29]:
shark_df.shape

(2649, 6)

In [30]:
shark_df.isnull().sum()

date          0
year          0
country       0
state         0
location      0
species     916
dtype: int64

In [31]:
shark_df['species'] = shark_df['species'].fillna('unknown').str.lower().str.strip()
contains_shark = shark_df['species'].str.contains('shark', na = False)
contains_question = shark_df['species'].str.contains(r'\?|female|not|involvement|or|possibly|juvenile', na = False)
shark_df.loc[~contains_shark | contains_question , 'species'] = 'unknown'
regex_limpieza = r'\d+\s*[m\']|[0-9,\'\.\[\]\"\(\)]|\bto\b|\ba\s+|\bfemale\b|\bmale\b|\bft\b'
regex_estetica = r'\s+'
shark_df['species'] = (shark_df['species']
    .str.replace(regex_limpieza, '', regex=True)
    .str.replace(regex_estetica, ' ', regex=True)
    .str.strip())
# cleaned_species = {'shark':'unknown', 'shark involvement not confirmed':'unknown','no shark involvement':'unknown',
#                    'shark involvement prior death was not confirmed':'unknown','small shark':'unknown',
#                    'blacktip or spinner shark':'unknown'}
# shark_df['species'] = shark_df['species'].replace(cleaned_species)
shark_df['species'].value_counts().head(20)

species
unknown                   1332
shark                      333
white shark                257
bull shark                 126
tiger shark                117
blacktip shark              39
bronze whaler shark         28
nurse shark                 27
small shark                 23
wobbegong shark             21
mako shark                  18
lemon shark                 16
raggedtooth shark           16
sandtiger shark             16
oceanic whitetip shark      13
reef shark                  12
blue shark                  10
grey reef shark             10
spinner shark               10
great white shark            8
Name: count, dtype: int64

In [32]:
shark_df['country'].value_counts().head(20)

country
usa                 1341
australia            576
south africa         152
brazil                64
new zealand           60
bahamas               55
mexico                37
new caledonia         31
french polynesia      30
egypt                 28
reunion               20
spain                 19
mozambique            11
fiji                  11
philippines           10
thailand              10
cuba                   9
papua new guinea       9
indonesia              8
ecuador                8
Name: count, dtype: int64

In [33]:
shark_df['state'].value_counts().head(30)

state
Florida                  691
New South Wales          190
Hawaii                   174
California               149
Western Australia        146
Queensland               109
South Carolina            94
North Carolina            78
Western Cape Province     57
Eastern Cape Province     53
South Australia           46
Pernambuco                45
Texas                     40
Victoria                  36
KwaZulu-Natal             34
North Island              30
South Island              18
New York                  18
Abaco Islands             18
NSW                       16
Oregon                    16
Alabama                   11
Tasmania                  10
Massachusetts              9
South Province             9
Quintana Roo               8
New Jersey                 8
North Province             8
Binh Dinh Province         8
Inhambane Province         7
Name: count, dtype: int64

In [34]:
shark_df['location'].value_counts().head(30)
                                        

location
New Smyrna Beach, Volusia County                                 150
Ponce Inlet, Volusia County                                       22
Cocoa Beach, Brevard  County                                      18
Myrtle Beach, Horry County                                        17
Melbourne Beach, Brevard County                                   15
Daytona Beach, Volusia County                                     12
Isle of Palms, Charleston County                                  11
Jacksonville Beach, Duval County                                  11
New Smyrna Beach                                                  10
Ponce Inlet, New Smyrna Beach, Volusia County                      9
Daytona Beach Shores, Volusia County                               8
Quy Nhon                                                           8
Florida Keys, Monroe County                                        7
Vero Beach, Indian River County                                    7
Ormond Beach, Volusia Cou

In [35]:
import re  

def extract_month_text(df):
    """
    Uses Regex to extract only the month names from the date column.
    """
    # Standardize to string and lowercase
    df['date'] = df['date'].astype(str).str.lower()

    def get_only_words(text):
        # We call the 're' module here
        words = re.findall(r'[a-z]+', text)
        
        # Filter strings shorter than 3 chars (like 'st', 'nd', 'rd')
        clean_words = [w for w in words if len(w) > 2]
        
        return clean_words[0] if clean_words else "unknown"

    # Apply and create the new column
    df['month'] = df['date'].apply(get_only_words)
    return df

In [36]:
shark_df = extract_month_text(shark_df)
shark_df.head()

,date,year,country,state,location,species,month
0,14/04,2026,maldives,Gaafu Alif Atoll,Kooddoo,unknown,unknown
1,3rd april,2026,australia,South Australia,Middleton Beach Fleurieu Peninsula Adelaide,unknown,april
2,26th march,2026,bahamas,Andros Island,Fresh Creek,unknown,march
3,25th march,2026,australia,South Australia,Cape Jaffa Limestone Coast,great white shark ft,march
4,22nd-23rd march,2026,australia,NSW,Little Avalon Beach,unknown,march


In [37]:
shark_df['month'].value_counts()

month
jul          311
sep          258
aug          236
jun          225
oct          217
apr          209
may          195
mar          169
jan          166
dec          156
nov          150
feb          133
reported      81
unknown       75
july          13
january       12
march          7
november       7
december       6
august         6
june           4
october        3
september      3
february       2
late           2
april          1
fall           1
summer         1
Name: count, dtype: int64

In [38]:
def normalize_months(df):

    month_map = {
        'jan': 'january',
        'feb': 'february',
        'mar': 'march',
        'apr': 'april',
        'may': 'may',
        'jun': 'june',
        'jul': 'july',
        'aug': 'august',
        'sep': 'september',
        'oct': 'october',
        'nov': 'november',
        'dec': 'december'
    }
    
    
    df['month'] = df['month'].replace(month_map)
    
    return df

shark_df = normalize_months(shark_df)

shark_df[['month']].value_counts()

month    
july         324
september    261
august       242
june         229
october      220
april        210
may          195
january      178
march        176
december     162
november     157
february     135
reported      81
unknown       75
late           2
fall           1
summer         1
Name: count, dtype: int64

In [39]:
print(shark_df.columns)

Index(['date', 'year', 'country', 'state', 'location', 'species', 'month'], dtype='str')


In [46]:
# 1. Standardize text to ensure a perfect match (lowercase and remove spaces)
shark_df['species'] = shark_df['species'].astype(str).str.lower().str.strip()

# 2. Define the list of values to be removed
values_to_remove = ['unknown', 'shark', 'small shark', 'kg shark']

# 3. Filter the dataframe: keep only rows where species is NOT in the list
# The '~' symbol means "NOT"
shark_df = shark_df[~shark_df['species'].isin(values_to_remove)]

print(f"New dataframe shape: {shark_df.shape}")
print(f"Remaining unique values in 'species': {shark_df['species'].nunique()}")

New dataframe shape: (958, 7)
Remaining unique values in 'species': 173


In [44]:
shark_df['species'].value_counts()

species
white shark                                            257
bull shark                                             126
tiger shark                                            117
blacktip shark                                          39
bronze whaler shark                                     28
                                                      ... 
cm cm shark                                              1
shark was seen in the area by witnesses                  1
thought involve -lb bull shark                           1
tiger sharks & bull sharks sharks in all                 1
miami cm blacktip shark and two cm bamboo catsharks      1
Name: count, Length: 174, dtype: int64

In [53]:

shark_df.duplicated().sum()

np.int64(0)

In [52]:
shark_df = shark_df.drop_duplicates()


In [54]:
# 1. Define the list of valid months we want to keep
valid_months = [
    'january', 'february', 'march', 'april', 'may', 'june', 
    'july', 'august', 'september', 'october', 'november', 'december'
]

# 2. Filter the dataframe to keep only rows where 'month' is in our list
# This automatically removes 'reported', 'unknown', 'late', 'fall', 'summer', etc.
shark_df = shark_df[shark_df['month'].isin(valid_months)]
print(shark_df['month'].value_counts())

month
july         103
january       81
october       81
september     81
august        79
june          79
april         74
may           72
december      67
march         56
november      56
february      54
Name: count, dtype: int64


In [65]:
# 1. Grouping by Country, Month, and Species
# I have removed the extra space after 'species' to match your Index
attacks_detailed = shark_df.groupby(['country', 'month', 'species']).size().reset_index(name='AttackCount')

# 2. Sorting the results
# Sorting alphabetically by country, and then by attack volume
attacks_detailed = attacks_detailed.sort_values(by=['country', 'AttackCount'], ascending=[True, False])

# 3. Final display
print("Detailed Shark Attacks (Cleaned Columns):")
display(attacks_detailed.head(50))

Detailed Shark Attacks (Cleaned Columns):


,country,month,species,AttackCount
47,australia,january,white shark,14
110,australia,september,white shark,14
20,australia,december,white shark,13
58,australia,july,white shark,13
39,australia,january,bull shark,10
101,australia,october,white shark,9
7,australia,april,white shark,6
12,australia,august,white shark,6
33,australia,february,white shark,6
65,australia,june,white shark,6


In [67]:
stats = shark_df[['country', 'month', 'species']].describe()

display(stats)

,country,month,species
count,883,883,883
unique,54,12,151
top,usa,july,white shark
freq,359,103,248


In [68]:
# 1. Create a summary table for categorical columns
# We select the columns of interest
top_stats = shark_df[['country', 'month', 'species']].describe().loc[['top', 'freq']]

# 2. Add a 'percentage' row to see how much the top value represents over the total
# (This gives you more context: Is it a clear winner or just by a little bit?)
total_rows = len(shark_df)
top_stats.loc['% of total'] = (top_stats.loc['freq'] / total_rows * 100).round(2).astype(str) + '%'

# 3. Transpose it (T) to make it look like a nice vertical table
summary_table = top_stats.T

print("Shark Attack Summary Table:")
display(summary_table)

Shark Attack Summary Table:


,top,freq,% of total
country,usa,359,40.66%
month,july,103,11.66%
species,white shark,248,28.09%


In [69]:
# Definir les columnes que volem analitzar
columns_to_analyze = ['country', 'month', 'species']
total_records = len(shark_df)

for col in columns_to_analyze:
    print(f"\n--- TOP 5 {col.upper()} ---")
    
    # 1. Calcular el recompte de valors (Top 5)
    counts = shark_df[col].value_counts().head(5)
    
    # 2. Calcular el percentatge
    percentages = (shark_df[col].value_counts(normalize=True).head(5) * 100).round(2)
    
    # 3. Combinar-ho en un DataFrame per mostrar-ho bonic
    top_5_df = pd.DataFrame({
        'Total Attacks': counts,
        'Percentage (%)': percentages.astype(str) + '%'
    })
    
    display(top_5_df)


--- TOP 5 COUNTRY ---


,Total Attacks,Percentage (%)
country,,
usa,359,40.66%
australia,252,28.54%
south africa,84,9.51%
new zealand,25,2.83%
bahamas,23,2.6%



--- TOP 5 MONTH ---


,Total Attacks,Percentage (%)
month,,
july,103,11.66%
january,81,9.17%
october,81,9.17%
september,81,9.17%
august,79,8.95%



--- TOP 5 SPECIES ---


,Total Attacks,Percentage (%)
species,,
white shark,248,28.09%
bull shark,114,12.91%
tiger shark,114,12.91%
blacktip shark,37,4.19%
bronze whaler shark,28,3.17%


In [72]:
# 1. Get the Top 5 countries with most attacks
top_5_countries = shark_df['country'].value_counts().head(5).index.tolist()

# 2. Filter the dataframe to include only these top 5 countries
top_df = shark_df[shark_df['country'].isin(top_5_countries)]

# 3. Group by country, month, and species to find the most common patterns
# We calculate the size of each group
top_patterns = top_df.groupby(['country', 'month', 'species']).size().reset_index(name='attack_count')

# 4. Sort and get the top result for each country
# We sort by country and attack_count (descending)
top_patterns = top_patterns.sort_values(['country', 'attack_count'], ascending=[True, False])

# 5. Keep only the #1 most frequent combination for each of the top 5 countries
final_top_stats = top_patterns.groupby('country').head(1)

print("The most frequent Month and Species for the Top 5 Countries:")
display(final_top_stats)


The most frequent Month and Species for the Top 5 Countries:


,country,month,species,attack_count
47,australia,january,white shark,14
118,bahamas,july,bull shark,4
137,new zealand,february,white shark,3
153,south africa,april,white shark,7
220,usa,august,white shark,14


In [73]:
# 1. Grouping by Country, State, and Location
# We count attacks and find the most frequent species for each spot
detailed_locations = shark_df.groupby(['country', 'state', 'location']).agg(
    total_attacks=('species', 'count'),
    most_frequent_species=('species', lambda x: x.value_counts().index[0] if not x.empty else 'unknown')
).reset_index()

# 2. Sorting by the places with the most attacks
detailed_locations = detailed_locations.sort_values(by='total_attacks', ascending=False)

# 3. Displaying the Top 10 "Hotspots" in the world
print("The 10 locations worldwide with the highest shark attack concentration:")
display(detailed_locations.head(10))

The 10 locations worldwide with the highest shark attack concentration:


,country,state,location,total_attacks,most_frequent_species
632,usa,Florida,"New Smyrna Beach, Volusia County",11,blacktip shark
603,usa,Florida,"Florida Keys, Monroe County",4,nurse shark
414,south africa,Eastern Cape Province,"Nahoon, East London",3,raggedtooth shark
592,usa,Florida,"Cocoa Beach, Brevard County",3,bull shark
604,usa,Florida,Fort Lauderdale,3,small nurse shark
685,usa,Hawaii,"Hanalei Bay, Kauai",3,tiger shark
294,egypt,"Hurghada, Red Sea Governorate",Sahl Hasheesh,2,oceaniic whitetip shark/tiger shark
195,australia,Western Australia,Exmouth,2,unknown shark ft
24,australia,New South Wales,Crescent Head,2,white shark
225,australia,Western Australia,Rottnest Island,2,white shark


In [ ]:
def get_country_top_analysis(df, country_name):
    # 1. Filtramos el país en minúscula
    country_df = df[df['country'] == country_name.lower()]
    
    # 2. Cogemos los 5 estados con más ataques
    top_states = country_df['state'].value_counts().head(5).index
    
    # 3. Filtrem el dataframe sólo con estos 5 estados
    final_df = country_df[country_df['state'].isin(top_states)]
    
    # 4. Agrupamos per State, Location y Species
    analysis = final_df.groupby(['state', 'location', 'species']).size().reset_index(name='attack_count')
    
    # 5. Ordenamos por estado y número de ataques
    analysis = analysis.sort_values(by=['state', 'attack_count'], ascending=[True, False])
    
    # 6. Para cada estado, nos quedamos con su "Top" (para no hacer la tabla infinita)
    # Mostraremos las 5 localizaciones/especies con más peso de cada estado top
    result = analysis.groupby('state').head(5)
    
    return result

In [75]:
print("--- TOP 5 ANALYSIS: USA ---")
usa_table = get_country_top_analysis(shark_df, 'usa')
display(usa_table)

--- TOP 5 ANALYSIS: USA ---


,state,location,species,attack_count
8,California,"Bunkers, Humboldt Bay, Eureka, Humboldt County",white shark,2
17,California,"Dillon Beach, Marin County",white shark,2
35,California,"Marina State Beach, Monterey County",white shark,2
62,California,Santa Barbara County,white shark,2
72,California,"Surf Beach, Lompoc, Santa Barbara County",white shark,2
139,Florida,"New Smyrna Beach, Volusia County",blacktip shark,6
93,Florida,"Cocoa Beach, Brevard County",bull shark,2
107,Florida,"Florida Keys, Monroe County",nurse shark,2
140,Florida,"New Smyrna Beach, Volusia County",bull shark,2
142,Florida,"New Smyrna Beach, Volusia County",spinner shark,2


In [76]:
print("--- TOP 5 ANALYSIS: AUSTRALIA ---")
aus_table = get_country_top_analysis(shark_df, 'australia')
display(aus_table)

--- TOP 5 ANALYSIS: AUSTRALIA ---


,state,location,species,attack_count
0,New South Wales,Angourie,white shark,1
1,New South Wales,"Belongil Beach, Byron Bay",white shark,1
2,New South Wales,Bermagui,mako shark,1
3,New South Wales,Birubi Point,wobbegong shark,1
4,New South Wales,Bondi Beach,bronze whaler shark,1
77,Queensland,"20 k off The Spit, off the Gold Coast",reef shark,1
78,Queensland,200 km east of Coolangatta,mako shark kg,1
79,Queensland,50 km east of Townsville,blacktip shark,1
80,Queensland,"Amity Point, North Stradbroke Island",bull shark,1
81,Queensland,Bargara Beach,tiger shark,1


In [77]:
print("--- TOP 5 ANALYSIS: SOUTH AFRICA ---")
sa_table = get_country_top_analysis(shark_df, 'south africa')
display(sa_table)

--- TOP 5 ANALYSIS: SOUTH AFRICA ---


,state,location,species,attack_count
8,Eastern Cape Province,East London,white shark,2
16,Eastern Cape Province,"Nahoon, East London",raggedtooth shark,2
21,Eastern Cape Province,"Second Beach, Port St. John's",tiger shark,2
0,Eastern Cape Province,2-3 km north of Sunday's River mouth,raggedtooth shark,1
1,Eastern Cape Province,"Albatros Point, near Jeffrey's Bay",white shark,1
27,KwaZulu-Natal,Alkantstrand,white shark,1
28,KwaZulu-Natal,Cape Vidal,raggedtooth shark -kg,1
29,KwaZulu-Natal,Durban,blacktip shark,1
30,KwaZulu-Natal,Eastmoor Crescent Beach,tiger shark,1
31,KwaZulu-Natal,Karridene,white shark,1


In [78]:
print("--- TOP 5 ANALYSIS: NEW ZEALAND ---")
nz_table = get_country_top_analysis(shark_df, 'new zealand')
display(nz_table)

--- TOP 5 ANALYSIS: NEW ZEALAND ---


,state,location,species,attack_count
0,Cook Islands,Beveridge Reef,grey reef shark,1
1,Mercury Islands,Great Mercury Island,bronze whaler shark,1
2,North Island,Baylys Beach,white shark,1
3,North Island,Cape Pallister,white shark,1
4,North Island,Cormandel,mako shark,1
5,North Island,Haumoana,broadnose sevengill shark,1
6,North Island,Hawkes Bay,mako shark,1
18,South Island,Clark Island,white shark,1
19,South Island,"Friendly Bay, Oamaru",sevengill shark,1
20,South Island,Garden Bay near Cosy Nook,-gill shark,1


In [79]:
print("--- TOP 5 ANALYSIS: BAHAMAS ---")
bah_table = get_country_top_analysis(shark_df, 'bahamas')
display(bah_table)

--- TOP 5 ANALYSIS: BAHAMAS ---


,state,location,species,attack_count
0,Grand Bahama Island,East End Point,reef shark,1
1,Abaco Islands,10 miles west of Walker’s Cay,bull shark,1
2,Abaco Islands,Allan-Pensacola Cay,bull shark,1
3,Abaco Islands,Carter's Cay,reef shark,1
4,Abaco Islands,"Coco Bay, Green Turtle Cay",nurse shark,1
5,Abaco Islands,Green Turtle Cay,bull shark,1
12,Freeport,Shark Junction,caribbean rreef shark,1
13,Grand Bahama Island,"High Rock, 25 to 30 miles east of Freeport",bull shark,1
14,Grand Bahama Island,Off West End,tiger sharks in area,1
15,Grand Bahama Island,Sweetings Cay,caribbean reef shark,1


In [82]:
# 1. Definimos la lista de tiburones
available_species = [
    'white shark', 'bull shark', 'tiger shark', 'blacktip shark', 
    'bronze whaler shark', 'nurse shark', 'wobbegong shark', 'mako shark', 
    'lemon shark', 'raggedtooth shark', 'sandtiger shark', 
    'oceanic whitetip shark', 'reef shark', 'blue shark', 
    'grey reef shark', 'spinner shark', 'great white shark'
]

def shark_locator(df):
    print("--- WELCOME TO THE SHARK LOCATOR ---")
    print(f"Available species: {', '.join(available_species)}")
    print("-" * 40)
    
    # Pedimos el input al usuario
    choice = input("Which shark are you looking for? ").lower().strip()
    
    if choice not in available_species:
        print(f"Sorry, '{choice}' is not in our top records. Please check the spelling.")
        return

    # 2. Filtramos el Df por la espécie escogida
    species_df = df[df['species'] == choice]
    
    if species_df.empty:
        print(f"No records found for {choice} after the cleaning process.")
        return

    # 3. Buscamos la localización con más ataques
    # Agrupamos, país, estado y localización
    best_spot = species_df.groupby(['country', 'state', 'location']).size().reset_index(name='count')
    best_spot = best_spot.sort_values('count', ascending=False).iloc[0]
    
    # 4. Resultat
    print(f"\nSTATISTICAL RESULT FOR: {choice.upper()}")
    print(f"If you want to see this shark, the place with the most records is:")
    print(f"📍 Country: {best_spot['country'].upper()}")
    print(f"📍 State:   {best_spot['state'].title()}")
    print(f"📍 Location: {best_spot['location'].title()}")
    print(f"Total historical records in this spot: {best_spot['count']}")

# LLamamos a la función
shark_locator(shark_df)

--- WELCOME TO THE SHARK LOCATOR ---
Available species: white shark, bull shark, tiger shark, blacktip shark, bronze whaler shark, nurse shark, wobbegong shark, mako shark, lemon shark, raggedtooth shark, sandtiger shark, oceanic whitetip shark, reef shark, blue shark, grey reef shark, spinner shark, great white shark
----------------------------------------

STATISTICAL RESULT FOR: BULL SHARK
If you want to see this shark, the place with the most records is:
📍 Country: BRAZIL
📍 State:   Pernambuco
📍 Location: Piedade Beach
Total historical records in this spot: 2
